#### download data from gee using python api

In [1]:
import ee
import geemap
import pandas as pd
import geopandas as gpd 
ee.Authenticate(auth_mode='localhost')
ee.Initialize(project='earth-engine-auth-project') 


In [34]:
### Lakes(vector date): chenghai(20240418), dianchi(20240415), erhai(20240801), 
### fuxian(20240415), lugu(20240523), qilu(20240415), xingyun(20240415), 
### yangzonghai(20240415), yilong(20240415)
lake_name = 'yilong' 
path_lake_vec = 'data/lakes-vec/'+lake_name+'_s2_20240415.gpkg' 


In [35]:
lake_gdf = gpd.read_file(path_lake_vec).to_crs(4326)
region_bounds = list(lake_gdf.bounds.iloc[0])    
region = ee.Geometry.Rectangle(region_bounds, 'EPSG:4326', False)   ## xmin, ymin, xmax, ymax 


In [36]:
dataset = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR').first()

date_start = '2023-01-01'
date_end = '2025-12-31'
dset = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR') \
        .filter(ee.Filter.date(date_start, date_end))
era5_pre = dset.select(['total_precipitation_sum'])

visualization = {
    # 'bands': ['temperature_2m'],
    'bands': ['total_precipitation_sum'],
    'min': 250,
    'max': 320,
    'palette': [
        '000080', '0000d9', '4000ff', '8000ff', '0080ff', '00ffff',
        '00ff80', '80ff00', 'daff00', 'ffff00', 'fff500', 'ffda00',
        'ffb000', 'ffa400', 'ff4f00', 'ff2500', 'ff0a00', 'ff00ff',
    ],
}

## visualization
empty = ee.Image().byte()
scene_outline = empty.paint(featureCollection=region, color=1, width=3)
map = geemap.Map()
map.centerObject(region, 11)
map.add_layer(era5_pre, visualization, 'climate', True, 0.8)
map.addLayer(scene_outline, {'palette': 'FF0000'}, 'region')
map


Map(center=[23.67442089130921, 102.5646271955453], controls=(WidgetControl(options=['position', 'transparent_b…

In [37]:
def calculate_regional_mean(image):
    stat = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=11132,       # ERA5-Land 的分辨率约为 11.1km
        maxPixels=1e9      
    )    
    date = ee.Date(image.get('system:time_start')).format('YYYY-MM')    
    return ee.Feature(None, stat).set('Date', date)

monthly_time_series = era5_pre.map(calculate_regional_mean) 
data = monthly_time_series.getInfo()['features']
df_data = [feat['properties'] for feat in data]
precip_df = pd.DataFrame(df_data).sort_values(by='Date')
precip_df.head()
precip_df.to_csv(f'data/climate/{lake_name}_era5_monthly_precipitation.csv', index=False) 
